# Podatkovne tabele (`DataFrame`)

**Podatkovni okvir** (`DataFrame`) je dvodimenzionalna podatkovna struktura knjižnice `pandas`. Hrani stolpce, ki imajo lahko različne podatkovne tipe, ter oznake vrstic in stolpcev. Pri tem predmetu ga bomo imenovali tudi **podatkovna tabela**.

Običajno v vsako vrstico shranimo podatke o eni opazovani enoti, v vsak stolpec pa vrednosti ene spremenljivke. V nadaljevanju bomo ustvarjali tabele, jim dodajali podatke in izbirali njihove dele.

## Ustvarjanje podatkovnih tabel

Tabelo ustvarimo s konstruktorjem `pd.DataFrame`. Če mu podamo slovar zaporedij, ključi določajo imena stolpcev, vrednosti pa podatke v stolpcih:

In [1]:
import pandas as pd

imena = ["Aleš", "Brina", "Ciril", "Darja", "Ema", "Filip"]
spoli = ["m", "ž", "m", "ž", "ž", "m"]

pt1 = pd.DataFrame({
    "ime": pd.Series(imena),
    "spol": pd.Series(spoli, dtype="category")
})
print(pt1)

     ime spol
0   Aleš    m
1  Brina    ž
2  Ciril    m
3  Darja    ž
4    Ema    ž
5  Filip    m


Obe zaporedji sta enako dolgi in imata isti privzeti indeks. Konstruktor ju **poravna po oznakah v indeksu**. Če oznake niso enake, uporabi njihovo unijo in manjkajoče vrednosti označi kot manjkajoče podatke.

Za vajo preveri, kaj se zgodi pri zaporedjih različnih dolžin. Nato namesto zaporedij podaj dva običajna seznama različnih dolžin in primerjaj rezultat.

Konstruktor sprejme tudi seznam seznamov, pri katerem vsak notranji seznam predstavlja vrstico. Imena stolpcev določimo z argumentom `columns`:

In [2]:
po_vrsticah = [[imena[i], spoli[i]] for i in range(len(imena))]
print(po_vrsticah)

pt2 = pd.DataFrame(po_vrsticah, columns=["ime", "spol"])
print(pt2)

[['Aleš', 'm'], ['Brina', 'ž'], ['Ciril', 'm'], ['Darja', 'ž'], ['Ema', 'ž'], ['Filip', 'm']]
     ime spol
0   Aleš    m
1  Brina    ž
2  Ciril    m
3  Darja    ž
4    Ema    ž
5  Filip    m


Atribut `shape` hrani dimenzije tabele, `size` pa število vseh celic, tudi tistih z manjkajočimi vrednostmi:

In [3]:
(nv, ns) = pt1.shape
print(f"Število vrstic v pt1 je {nv}, število stolpcev {ns}, število vrednosti pa {pt1.size}.")

Število vrstic v pt1 je 6, število stolpcev 2, število vrednosti pa 12.


Tabela `pt1` ima šest vrstic, dva stolpca in dvanajst celic.

Nov stolpec dodamo s prirejanjem na ime stolpca. Za vsako osebo naključno izberemo višino v centimetrih med 150 in 220, vključno z obema mejama:

In [4]:
from random import randint, seed

seed(42)

pt1["visina"] = pd.Series(
    [randint(150, 220) for _ in range(len(pt1))], index=pt1.index
)
print(pt1)

     ime spol  visina
0   Aleš    m     164
1  Brina    ž     153
2  Ciril    m     185
3  Darja    ž     181
4    Ema    ž     178
5  Filip    m     167


Če stolpec z izbranim imenom že obstaja, prirejanje nadomesti njegove vrednosti. Pri prirejanju zaporedja se podatki poravnajo po oznakah vrstic. Pri prirejanju seznama pa se vrednosti vstavijo po položajih; njegova dolžina mora ustrezati številu vrstic.

Stolpec `visina` izberemo z `pt1["visina"]`. Dostop `pt1.visina` je prav tako mogoč, če je ime veljavno ime atributa in ne sovpada z obstoječim atributom ali metodo. Zapis z oglatimi oklepaji je zato splošnejši in ga bomo uporabljali v nadaljevanju:

In [5]:
pt2["visina"] = pt1["visina"]
print(pt2)

     ime spol  visina
0   Aleš    m     164
1  Brina    ž     153
2  Ciril    m     185
3  Darja    ž     181
4    Ema    ž     178
5  Filip    m     167


Dodajmo še datume rojstva, naključno izbrane med 1. januarjem 2001 in 31. decembrom 2002, vključno z obema mejama:

In [6]:
datum_od = pd.to_datetime("2001-01-01")
datum_do = pd.to_datetime("2002-12-31")

# Število dni v obdobju, vključno z obema mejama.
ndni = (datum_do - datum_od).days + 1

# randint vključuje obe meji; največji dovoljeni odmik je ndni - 1.
casovne_razlike = pd.to_timedelta(
    [randint(0, ndni - 1) for _ in range(len(pt1))], unit="D"
)
datumi = datum_od + casovne_razlike

pt1["datum_rojstva"] = datumi
pt2["datum_rojstva"] = datumi
print(pt1)

     ime spol  visina datum_rojstva
0   Aleš    m     164    2001-04-15
1  Brina    ž     153    2002-11-24
2  Ciril    m     185    2002-07-13
3  Darja    ž     181    2001-03-31
4    Ema    ž     178    2002-08-28
5  Filip    m     167    2002-03-09


Pred dodajanjem novih vrstic spoznajmo še druge atribute tabele.

## Atributi podatkovnih tabel

| Atribut | Opis |
|:---|:---|
| `shape` | Par (število vrstic, število stolpcev) |
| `size` | Število vseh celic v tabeli |
| `columns` | Oznake stolpcev kot objekt `Index` |
| `dtypes` | Zaporedje podatkovnih tipov stolpcev |
| `index` | Oznake vrstic kot objekt `Index` |
| `values` | Polje tipa `numpy.ndarray` z vrednostmi celic |
| `at` in `iat` | Izbira posamezne celice po oznakah oziroma položajih |
| `loc` in `iloc` | Izbira delov tabele po oznakah oziroma položajih |

Nekatere atribute poznamo že iz zaporedij. Oglejmo si `columns` in `dtypes`:

In [7]:
print(pt1.columns)
print(pt1.dtypes)

Index(['ime', 'spol', 'visina', 'datum_rojstva'], dtype='object')
ime                      object
spol                   category
visina                    int64
datum_rojstva    datetime64[ns]
dtype: object


Atribut `columns` vrne objekt `Index` z imeni stolpcev. Atribut `dtypes` vrne zaporedje, katerega oznake so imena stolpcev, vrednosti pa njihovi podatkovni tipi. Oznake stolpcev z metodo `tolist()` pretvorimo v seznam:

In [8]:
print(pt1.columns.tolist())

['ime', 'spol', 'visina', 'datum_rojstva']


Atribut `index` hrani oznake vrstic. Dolžina indeksa je enaka številu vrstic, dolžina `columns` pa številu stolpcev:

In [9]:
print((len(pt1.index), len(pt1.columns)) == pt1.shape)

True


Zmnožek števila vrstic in števila stolpcev je enak številu celic:

In [10]:
print(pt1.shape[0] * pt1.shape[1] == pt1.size)

True


Atribut `values` vrne dvodimenzionalno polje tipa `numpy.ndarray`. Vsi elementi polja imajo skupen podatkovni tip; pri naši tabeli z različnimi tipi stolpcev je to `object`. Polje dobimo tudi z metodo `to_numpy()`:

In [11]:
print(pt1.values)

[['Aleš' 'm' 164 Timestamp('2001-04-15 00:00:00')]
 ['Brina' 'ž' 153 Timestamp('2002-11-24 00:00:00')]
 ['Ciril' 'm' 185 Timestamp('2002-07-13 00:00:00')]
 ['Darja' 'ž' 181 Timestamp('2001-03-31 00:00:00')]
 ['Ema' 'ž' 178 Timestamp('2002-08-28 00:00:00')]
 ['Filip' 'm' 167 Timestamp('2002-03-09 00:00:00')]]


## Indeksiranje podatkovnih tabel

Atributa `loc` in `iloc` poznamo že iz zaporedij. Pri tabeli lahko določimo izbiro tako vrstic kot stolpcev: `pt.loc[vrstice, stolpci]`.

Pri `loc` uporabimo oznake, sezname oznak, rezine ali logične pogoje. Znak `:` izbere vse vrstice oziroma stolpce. Rezine po oznakah vključujejo obe meji, tudi kadar so oznake cela števila:

In [12]:
print(pt1.loc[:, "ime"])
print(pt1.loc[:, ["ime"]])
print(pt1.loc[2:0:-1, "ime":"visina"])
print(pt1.loc[[0, 3, 3, 5], ["ime", "visina"]])

0     Aleš
1    Brina
2    Ciril
3    Darja
4      Ema
5    Filip
Name: ime, dtype: object
     ime
0   Aleš
1  Brina
2  Ciril
3  Darja
4    Ema
5  Filip
     ime spol  visina
2  Ciril    m     185
1  Brina    ž     153
0   Aleš    m     164
     ime  visina
0   Aleš     164
3  Darja     181
3  Darja     181
5  Filip     167


Prvi zapis vrne zaporedje, drugi pa tabelo z enim stolpcem. Pri rezini `2:0:-1` izberemo vrstice z oznakami 2, 1 in 0 v tem vrstnem redu; rezina `"ime":"visina"` vključuje oba mejna stolpca.

Za vajo pojasni, zakaj neposredna primerjava rezultatov prvih dveh zapisov sproži izjemo. Izjemo tukaj prestrežemo, da lahko nadaljujemo izvajanje:

In [13]:
try:
    pt1.loc[:, "ime"] == pt1.loc[:, ["ime"]]
except ValueError as napaka:
    print(f"ValueError: {napaka}")

ValueError: Operands are not aligned. Do `left, right = left.align(right, axis=1, copy=False)` before operating.


Pri primerjavi tabele z zaporedjem se oznake zaporedja privzeto primerjajo z oznakami **stolpcev** tabele. V našem primeru so oznake zaporedja oznake vrstic, zato se ne ujemajo. Za vajo primerjaj zaporedje s stolpcem, izbranim iz enostolpčne tabele, in preveri rezultat.

Vrstice lahko izberemo tudi s seznamom logičnih vrednosti. Seznam mora imeti eno vrednost za vsako vrstico:

In [14]:
print(pt1.loc[[False, True, False, False, True, False], :])

     ime spol  visina datum_rojstva
1  Brina    ž     153    2002-11-24
4    Ema    ž     178    2002-08-28


Logične vrednosti običajno dobimo s primerjavo stolpca in posamezne vrednosti (skalarja):

In [15]:
print(pt1["spol"] == "m")

0     True
1    False
2     True
3    False
4    False
5     True
Name: spol, dtype: bool


Primerjava vrne logično zaporedje, ki ga uporabimo kot pogoj za izbiro vrstic. Tako izpišemo imena oseb moškega spola:

In [16]:
print(pt1.loc[pt1["spol"] == "m", "ime"].tolist())

['Aleš', 'Ciril', 'Filip']


Pogoj izbere vrstice, ime stolpca pa podatke, ki jih želimo izpisati. Rezultat z metodo `tolist()` pretvorimo v seznam.

Podobno izberemo podatke za osebe, rojene leta 2002 ali pozneje:

In [17]:
print(pt1.loc[pt1["datum_rojstva"] >= pd.Timestamp("2002-01-01"), :])

     ime spol  visina datum_rojstva
1  Brina    ž     153    2002-11-24
2  Ciril    m     185    2002-07-13
4    Ema    ž     178    2002-08-28
5  Filip    m     167    2002-03-09


Atribut `iloc` izbira po položajih vrstic in stolpcev. Rezine imajo enak pomen kot pri seznamih: zgornja meja ni vključena. Naslednji zapis izbere vrstice od položaja 2 do predzadnje vrstice in prva dva stolpca:

In [18]:
print(pt1.iloc[2:-1, :2])

     ime spol
2  Ciril    m
3  Darja    ž
4    Ema    ž


Novo vrstico lahko dodamo s prirejanjem prek `loc` na oznako, ki je še ni v indeksu. V naši tabeli so oznake zaporedna cela števila od 0, zato je število vrstic ustrezna nova oznaka. Pri drugačnem indeksu to ne velja nujno; prirejanje na obstoječo oznako bi vrstico nadomestilo. Atribut `iloc` tabele ne more povečati:

In [19]:
nova_vrstica = ["Gregor", "m", 181, pd.to_datetime("2002-07-29")]
pt1.loc[pt1.shape[0]] = nova_vrstica
pt2.loc[pt2.shape[0]] = nova_vrstica
print(pt1)

      ime spol  visina datum_rojstva
0    Aleš    m     164    2001-04-15
1   Brina    ž     153    2002-11-24
2   Ciril    m     185    2002-07-13
3   Darja    ž     181    2001-03-31
4     Ema    ž     178    2002-08-28
5   Filip    m     167    2002-03-09
6  Gregor    m     181    2002-07-29


Za vajo preuči metodo `drop` in z njo odstrani stolpec ter vrstico. Preveri, ali metoda privzeto spremeni izvirno tabelo ali vrne novo. Po dodajanju vrstice preveri tudi `dtypes`: pri spreminjanju tabele se lahko podatkovni tipi stolpcev spremenijo.

Atributa `at` in `iat` sta namenjena izbiri posamezne celice. Prvi uporablja oznaki vrstice in stolpca, drugi pa njuna položaja:

In [20]:
print(pt1.at[1, "ime"])
print(pt1.iat[-1, -1])

Brina
2002-07-29 00:00:00


---

## Poravnava indeksov

**Poravnava** (angl. _alignment_) je samodejno usklajevanje oznak v indeksih zaporedij in tabel. Pri združevanju zaporedij v tabelo se vrednosti z isto oznako znajdejo v isti vrstici. Če zaporedje nima vrednosti za določeno oznako, je ustrezna celica označena kot manjkajoča, na primer z `NaN`.

### Samodejna poravnava

Ustvarimo časovno vrsto za prvih deset dni oktobra 2026:

In [ ]:
from random import uniform

seed(42)

datumi = pd.date_range("2026-10-01", freq="D", periods=10)
vrednosti = [uniform(-1, 1) for _ in range(len(datumi))]
z0 = pd.Series(vrednosti, index=datumi)
print(z0)

2023-10-01    0.278854
2023-10-02   -0.949978
2023-10-03   -0.449941
2023-10-04   -0.553579
2023-10-05    0.472942
2023-10-06    0.353399
2023-10-07    0.784359
2023-10-08   -0.826122
2023-10-09   -0.156156
2023-10-10   -0.940406
Freq: D, dtype: float64


Sestavimo tabelo s stolpci `Z_t`, `Z_t_1` in `Z_t_2`. Vrstica z datumom $t$ naj vsebuje vrednosti časovne vrste na dan $t$, naslednji dan $t+1$ in dan $t+2$.

Za prvi stolpec uporabimo `z0`. Za druga dva najprej poskusimo izpustiti prvi element oziroma prva dva elementa zaporedja:

In [22]:
def zamik(z):
    return z.iloc[1:]


z1 = zamik(z0)
z2 = zamik(z1)
pt_z = pd.DataFrame({"Z_t": z0, "Z_t_1": z1, "Z_t_2": z2})
print(pt_z)

                 Z_t     Z_t_1     Z_t_2
2023-10-01  0.278854       NaN       NaN
2023-10-02 -0.949978 -0.949978       NaN
2023-10-03 -0.449941 -0.449941 -0.449941
2023-10-04 -0.553579 -0.553579 -0.553579
2023-10-05  0.472942  0.472942  0.472942
2023-10-06  0.353399  0.353399  0.353399
2023-10-07  0.784359  0.784359  0.784359
2023-10-08 -0.826122 -0.826122 -0.826122
2023-10-09 -0.156156 -0.156156 -0.156156
2023-10-10 -0.940406 -0.940406 -0.940406


Rezultat ne ustreza želeni tabeli. Prva vrstica bi morala vsebovati prve tri vrednosti časovne vrste, vendar sta drugi in tretji stolpec v njej manjkajoča.

Razlog je poravnava: rezina ohrani oznake elementov, zato konstruktor vrednosti poravna po prvotnih datumih. Oglejmo si rezultat funkcije `zamik`:

In [23]:
print(zamik(z0))

2023-10-02   -0.949978
2023-10-03   -0.449941
2023-10-04   -0.553579
2023-10-05    0.472942
2023-10-06    0.353399
2023-10-07    0.784359
2023-10-08   -0.826122
2023-10-09   -0.156156
2023-10-10   -0.940406
Freq: D, dtype: float64


Vrednost za 2. oktober še vedno nosi oznako `2023-10-02`. Če jo želimo prikazati kot vrednost naslednjega dne v vrstici za 1. oktober, ji moramo prirediti oznako `2023-10-01`.

Funkcijo popravimo tako, da vrednostim brez prvega elementa priredimo izvirni indeks brez zadnje oznake. Ustvarimo tudi kopijo rezine:

In [24]:
def zamik(z):
    zz = z.iloc[1:].copy()
    zz.index = z.index[:-1]
    return zz


print(zamik(z0))

2023-10-01   -0.949978
2023-10-02   -0.449941
2023-10-03   -0.553579
2023-10-04    0.472942
2023-10-05    0.353399
2023-10-06    0.784359
2023-10-07   -0.826122
2023-10-08   -0.156156
2023-10-09   -0.940406
Freq: D, dtype: float64


Zdaj ponovno sestavimo tabelo:

In [25]:
z1 = zamik(z0)
z2 = zamik(z1)

pt_z = pd.DataFrame({
    "Z_t": z0,
    "Z_t_1": z1,
    "Z_t_2": z2
})
print(pt_z)

                 Z_t     Z_t_1     Z_t_2
2023-10-01  0.278854 -0.949978 -0.449941
2023-10-02 -0.949978 -0.449941 -0.553579
2023-10-03 -0.449941 -0.553579  0.472942
2023-10-04 -0.553579  0.472942  0.353399
2023-10-05  0.472942  0.353399  0.784359
2023-10-06  0.353399  0.784359 -0.826122
2023-10-07  0.784359 -0.826122 -0.156156
2023-10-08 -0.826122 -0.156156 -0.940406
2023-10-09 -0.156156 -0.940406       NaN
2023-10-10 -0.940406       NaN       NaN


Zadnja vrstica nima vrednosti za naslednja dva dneva, predzadnja pa nima vrednosti za dan $t+2$. Teh opazovanj v izvirni časovni vrsti ni, zato so ustrezne celice manjkajoče.

Enak rezultat lahko dobimo z metodama `z0.shift(-1)` in `z0.shift(-2)`, ki premakneta vrednosti, indeks pa ohranita. Namesto krajših zaporedij vrneta zaporedji izvirne dolžine z manjkajočimi vrednostmi na koncu. Pri neenakomerno razporejenih datumih tak premik pomeni naslednje **opazovanje**, ne nujno naslednji dan.

### Izrecna poravnava

Metoda [`align`](https://pandas.pydata.org/docs/reference/api/pandas.Series.align.html) vrne dve zaporedji s skupnim indeksom. Z argumentom `join` določimo, katere oznake uporabimo. To je koristno, kadar imata zaporedji različne oznake:

In [26]:
z1 = pd.Series([1, 2, 3, 4], index=[3, 5, 2, 0])
z2 = pd.Series([1, 2, 3, 4], index=[5, 0, 1, 4])

Zaporedji najprej neposredno združimo v tabelo:

In [27]:
pt = pd.DataFrame({"Z1": z1, "Z2": z2})
print(pt)

    Z1   Z2
0  4.0  2.0
1  NaN  3.0
2  3.0  NaN
3  1.0  NaN
4  NaN  4.0
5  2.0  1.0


Konstruktor uporabi vse oznake iz obeh zaporedij. Kjer ustrezna vrednost manjka, se pojavi `NaN`.

Z metodo `align` lahko način poravnave izberemo izrecno. Argument `join` ima štiri možne vrednosti:

  * `"left"`: oznake prvega zaporedja v njegovem vrstnem redu;
  * `"right"`: oznake drugega zaporedja v njegovem vrstnem redu;
  * `"inner"`: skupne oznake v vrstnem redu prvega zaporedja;
  * `"outer"` (privzeto): unija oznak, urejena po vrednosti oznak.

Oglejmo si vse štiri možnosti:

In [28]:
z1p, z2p = z1.align(z2, join="left")
pt = pd.DataFrame({"Z1": z1p, "Z2": z2p})
print(pt)

   Z1   Z2
3   1  NaN
5   2  1.0
2   3  NaN
0   4  2.0


In [29]:
z1p, z2p = z1.align(z2, join="right")
pt = pd.DataFrame({"Z1": z1p, "Z2": z2p})
print(pt)

    Z1  Z2
5  2.0   1
0  4.0   2
1  NaN   3
4  NaN   4


In [30]:
z1p, z2p = z1.align(z2, join="inner")
pt = pd.DataFrame({"Z1": z1p, "Z2": z2p})
print(pt)

   Z1  Z2
5   2   1
0   4   2


In [31]:
z1p, z2p = z1.align(z2, join="outer")
pt = pd.DataFrame({"Z1": z1p, "Z2": z2p})
print(pt)

    Z1   Z2
0  4.0  2.0
1  NaN  3.0
2  3.0  NaN
3  1.0  NaN
4  NaN  4.0
5  2.0  1.0


Pri naših zaporedjih neposredno združevanje v tabelo da enak rezultat kot poravnava z `join="outer"`.

### Ponovno indeksiranje

Metoda [`reindex`](https://pandas.pydata.org/docs/reference/api/pandas.Series.reindex.html) vrne zaporedje z izbranim novim indeksom. Privzeto pri novih oznakah brez ustrezne vrednosti vstavi manjkajoče podatke.

Z nastavitvijo `method="nearest"` za novo oznako brez natančnega ujemanja uporabimo vrednost pri najbližji izvirni oznaki. To **ni isto kot poravnava natančnih ujemanj**: izberemo približno ujemanje. Izvirni indeks mora biti urejen in brez ponovitev, razdalje med oznakami pa morajo biti smiselno določene, na primer pri številih ali datumih. V našem primeru indeks najprej uredimo z metodo `sort_index`:

In [32]:
z1s = z1.sort_index()
print(f"Zaporedje pred ponovnim indeksiranjem:\n{z1s}")
z1p = z1s.reindex(z2.index, method="nearest")
print(f"Novi indeks: {z2.index}")
print(f"Zaporedje po ponovnem indeksiranju:\n{z1p}")

Zaporedje pred ponovnim indeksiranjem:
0    4
2    3
3    1
5    2
dtype: int64
Novi indeks: Index([5, 0, 1, 4], dtype='int64')
Zaporedje po ponovnem indeksiranju:
5    2
0    4
1    3
4    2
dtype: int64


Rezultat ima enake oznake in njihov vrstni red kot `z2`. Nova oznaka 1 prevzame vrednost 3 pri izvirni oznaki 2. Nova oznaka 4 je enako oddaljena od izvirnih oznak 3 in 5; pri izenačeni razdalji `nearest` izbere **večjo oznako**, zato prevzame vrednost 2 pri oznaki 5.

Metoda ne premakne izvirnih oznak, temveč za vsako novo oznako poišče ustrezno vrednost. Tako lahko isto izvirno vrednost uporabi večkrat. Z argumentom `tolerance` lahko omejimo dovoljeno razdaljo; če v tej razdalji ni ujemanja, dobimo manjkajočo vrednost.

Dobljeno zaporedje združimo z `z2`:

In [33]:
pt = pd.DataFrame({"Z1": z1p, "Z2": z2})
print(pt)

   Z1  Z2
5   2   1
0   4   2
1   3   3
4   2   4


Za vajo ponovno indeksiraj `z2` po oznakah `z1`. Pred tem uredi izvirni indeks. Ali dobiš enak rezultat? Pojasni, zakaj smer ponovnega indeksiranja vpliva na izbiro vrednosti.

Nekatere metode, na primer `sort_index`, sprejmejo argument `inplace=True`. V tem primeru spremenijo objekt, na katerem jih pokličemo, in vrnejo `None`. To ne zagotavlja manjše porabe pomnilnika, saj lahko metoda še vedno ustvari vmesne podatke. Metoda `reindex` takega argumenta nima in vrne nov objekt.

Naslednji primer uredi `z1` neposredno in nato ustvari ponovno indeksirano zaporedje:

In [34]:
z1.sort_index(inplace=True)
z1p = z1.reindex(z2.index, method="nearest")
pt = pd.DataFrame({"Z1": z1p, "Z2": z2})
print(pt)

   Z1  Z2
5   2   1
0   4   2
1   3   3
4   2   4
